[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_05/MFM_ch5_inference/MFM_ch5_inference.ipynb)

# MFM_ch5_inference

Asymptotic inference for GARCH models on the S&P 500, Bitcoin and the BET: robust versus classical variance of the Gaussian QMLE (sandwich versus inverse-Hessian standard errors and sqrt((kappa_z - 1)/2)), boundary likelihood ratio tests (Normal versus Student-t), delta-method standard errors of long-run volatility and half-life, moments and autocorrelation of squared shocks, the squared-residual portmanteau with estimated parameters (Li-Mak), GJR persistence under skewed-t innovations, and Giacomini-White tests (rolling 1000-day window) with the Model Confidence Set on QLIKE.

*Modelling Financial Markets (MFM), Chapter 5: Conditional Volatility, GARCH Models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (Kevin Sheppard) is not preinstalled on Colab
try:
    import arch
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats, optimize
from scipy.signal import lfilter
from scipy.special import gammaln
from arch import arch_model
from arch.univariate import ARCHInMean, ConstantMean, GARCH, FIGARCH, StudentsT, ZeroMean
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'btc': Amber, 'eurron': Forest, 'gold': Purple}

SEED = 42
TABLE_DIR = '.'
NUM = {}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def acf(x, nlags):
    x = np.asarray(x) - np.mean(x)
    d = np.sum(x ** 2)
    return np.array([np.sum(x[k:] * x[:-k]) / d for k in range(1, nlags + 1)])

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, eticheta, grup, data de start)
MARKETS = {
    'sp500':  ('GSPC.INDX',      'S&P 500',                 'Equity', '2000-01-01'),
    'bet':    ('BET',            'BET',                     'Equity', '2000-01-01'),
    'bettr':  ('BETTR.INDX',     'BET-TR',                  'Equity', '2014-09-23'),
    'btc':    ('BTC-USD.CC',     'Bitcoin',                 'Crypto', '2014-09-17'),
    'eurron': ('REF:EUR',        'EUR/RON (BNR reference)', 'FX',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX',   'Gold (XAU/USD)',          'Commodity', '2000-01-01'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Pretul de inchidere zilnic, curatat dupa conventiile capitolului."""
    symbol, _, group, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # fara cotatii de weekend
        s = s[s.diff() != 0]                  # fara sarbatori completate cu pretul anterior
    return s.rename(name)


def pct_returns(name, start=None, end=END):
    """Randamente log zilnice, in procente, pe calendarul propriu al seriei."""
    return (100 * np.log(load_close(name, start, end)).diff().dropna()).rename(name)


def periods_per_year(r):
    """Numarul mediu de observatii pe an calendaristic (frecventa reala a seriei)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def load_vix(start='2000-01-01', end=END):
    """Indicele VIX (volatilitatea implicita pe 30 de zile a S&P 500, in procente anualizate)."""
    s = read_market('VIX.INDX')['close'].loc[start:end]
    return s[s.index.dayofweek < 5].rename('vix')


rets = {k: pct_returns(k) for k in MARKETS}
PPY = {k: periods_per_year(rets[k]) for k in MARKETS}
ORDER = list(MARKETS)
MAIN = ['sp500', 'bet', 'btc', 'eurron', 'gold']
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs',
          round(PPY[k], 1), 'obs/year')

## GARCH tools

In [ ]:
VOL_SPEC = {'GARCH': dict(vol='GARCH', p=1, o=0, q=1),
            'GJR': dict(vol='GARCH', p=1, o=1, q=1),
            'EGARCH': dict(vol='EGARCH', p=1, o=1, q=1)}
DISTS = {'normal': 'Normal', 't': 'Student-t', 'skewt': 'Skewed-t', 'ged': 'GED'}


class Fit:
    """Rezultatul unei estimari, exprimat in unitatile seriei originale (randamente in %)."""

    def __init__(self, res, c, r, vol):
        self.res, self.c, self.r, self.vol = res, c, r, vol
        p = res.params.copy()
        p['mu'] = p['mu'] / c
        if vol == 'EGARCH':
            p['omega'] = p['omega'] - (1 - p['beta[1]']) * np.log(c ** 2)
        else:
            p['omega'] = p['omega'] / c ** 2
        self.params = p
        # covarianta robusta (Bollerslev-Wooldridge) transformata in unitatile originale cu jacobianul
        # transformarii: mu/c, omega/c^2 (GARCH, GJR) sau omega - (1 - beta) ln c^2 (EGARCH)
        names = list(res.params.index)
        J = np.eye(len(names))
        J[names.index('mu'), names.index('mu')] = 1 / c
        if vol == 'EGARCH':
            J[names.index('omega'), names.index('beta[1]')] = np.log(c ** 2)
        else:
            J[names.index('omega'), names.index('omega')] = 1 / c ** 2
        self.cov = pd.DataFrame(J @ res.param_cov.values @ J.T, index=names, columns=names)
        self.se = pd.Series(np.sqrt(np.diag(self.cov.values)), index=names)
        self.tvalues = p[names] / self.se
        self.pvalues = pd.Series(2 * stats.norm.sf(np.abs(self.tvalues.values)), index=names)
        n = res.nobs
        self.nobs = n
        self.loglik = res.loglikelihood + n * np.log(c)
        k = len(p)
        self.aic = -2 * self.loglik + 2 * k
        self.bic = -2 * self.loglik + k * np.log(n)
        self.sigma = res.conditional_volatility / c
        self.z = res.std_resid
        self.eps = res.resid / c                   # reziduuri nestandardizate, in unitatile seriei originale
        self.converged = res.convergence_flag == 0

    @property
    def persistence(self):
        p = self.params
        if self.vol == 'EGARCH':
            return p['beta[1]']
        g = p.get('gamma[1]', 0.0)
        return p['alpha[1]'] + p['beta[1]'] + g * self.neg_share()

    def neg_share(self):
        """E[z^2 1(z < 0)] pentru distributia inovatiilor (0.5 daca este simetrica); intra in persistenta GJR."""
        d = self.res.model.distribution
        if d.name.startswith('Standardized Skew'):
            from scipy import integrate
            par = self.res.params[d.parameter_names()].values
            pdf = lambda x: float(np.exp(d.loglikelihood(par, np.array([x]), np.array([1.0]), individual=True))[0])
            return float(integrate.quad(lambda x: x ** 2 * pdf(x), -np.inf, 0, limit=200)[0])
        return 0.5

    def uncond_var(self):
        p = self.params
        if self.vol == 'EGARCH':
            return np.nan
        return p['omega'] / (1 - self.persistence)

    def forecast_var(self, horizon):
        """Prognoza variantei zilnice pentru t+1, ..., t+h, in %^2."""
        f = self.res.forecast(horizon=horizon, reindex=False, method='analytic' if self.vol != 'EGARCH' else 'simulation',
                              simulations=2000)
        return f.variance.iloc[-1].values / self.c ** 2


def scale_for(r):
    """Factor de rescalare (putere a lui 10) astfel incat abaterea standard sa fie de ordinul 1."""
    s = r.std()
    return 10.0 if s < 0.5 else 1.0


def fit_garch(r, vol='GARCH', dist='t', mean='Constant', last_obs=None, c=None):
    c = scale_for(r) if c is None else c
    am = arch_model(c * r, mean=mean, dist=dist, **VOL_SPEC[vol])
    res = am.fit(disp='off', last_obs=last_obs, options={'maxiter': 1000})
    return Fit(res, c, r, vol)


def half_life(persistence):
    """Numarul de zile dupa care jumatate din socul asupra variantei s-a disipat."""
    return np.log(0.5) / np.log(persistence) if 0 < persistence < 1 else np.inf


def news_impact(fit, eps, sigma2_bar=None):
    """sigma^2_t ca functie de socul eps_{t-1}, cu sigma^2_{t-1} fixat la varianta de selectie."""
    p = fit.params
    s2 = fit.r.var() if sigma2_bar is None else sigma2_bar
    if fit.vol == 'EGARCH':
        z = eps / np.sqrt(s2)
        return np.exp(p['omega'] + p['alpha[1]'] * (np.abs(z) - np.sqrt(2 / np.pi)) + p['gamma[1]'] * z
                      + p['beta[1]'] * np.log(s2))
    g = p.get('gamma[1]', 0.0)
    return p['omega'] + (p['alpha[1]'] + g * (eps < 0)) * eps ** 2 + p['beta[1]'] * s2


def sign_bias_test(z, eps):
    """Engle-Ng (1993, ec. 18): z_t^2 pe S-_{t-1}, S-_{t-1} eps_{t-1}, S+_{t-1} eps_{t-1}, cu eps_{t-1} reziduul
    NESTANDARDIZAT si z_t reziduul standardizat; t-uri si testul comun T R^2 ~ chi2(3)."""
    d = pd.concat([pd.Series(z), pd.Series(eps)], axis=1, keys=['z', 'e']).dropna()
    z, e = d['z'].values, d['e'].values
    y = z[1:] ** 2
    el = e[:-1]
    sneg = (el < 0).astype(float)
    X = np.column_stack([np.ones_like(el), sneg, sneg * el, (1 - sneg) * el])
    ols = sm.OLS(y, X).fit()
    joint = len(y) * ols.rsquared
    return {'sign_t': ols.tvalues[1], 'neg_size_t': ols.tvalues[2], 'pos_size_t': ols.tvalues[3],
            'joint': joint, 'joint_p': 1 - stats.chi2.cdf(joint, 3)}


def ljung_box(x, lags=10):
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb = acorr_ljungbox(pd.Series(np.asarray(x)).dropna(), lags=[lags])
    return float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])


def arch_lm(x, lags=5):
    from statsmodels.stats.diagnostic import het_arch
    lm, lmp, _, _ = het_arch(pd.Series(np.asarray(x)).dropna(), nlags=lags)
    return float(lm), float(lmp)


def ewma_variance(r, lam=0.94, init=250):
    """Prognoza EWMA pentru ziua t+1 (aliniata la t+1): s2_{t+1} = lam s2_t + (1 - lam) r_t^2."""
    x = r.values
    s2 = np.empty(len(x) + 1)
    s2[0] = np.mean(x[:init] ** 2)
    for t in range(len(x)):
        s2[t + 1] = lam * s2[t] + (1 - lam) * x[t] ** 2
    return pd.Series(s2[1:-1], index=r.index[1:])


def qlike(proxy, h):
    """Pierderea QLIKE (Patton, 2011), forma folosita la comparatii: proxy/h + ln h."""
    return proxy / h + np.log(h)


def mz_regression(proxy, h, lags=5):
    """Mincer-Zarnowitz: proxy = a + b h + u, erori HAC; testul Wald a = 0, b = 1."""
    X = sm.add_constant(np.asarray(h))
    ols = sm.OLS(np.asarray(proxy), X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    w = ols.wald_test((np.eye(2), np.array([0.0, 1.0])), scalar=True)
    return {'a': ols.params[0], 'b': ols.params[1], 'a_se': ols.bse[0], 'b_se': ols.bse[1],
            'r2': ols.rsquared, 'wald': float(w.statistic), 'wald_p': float(w.pvalue)}


def dm_test(loss_a, loss_b, lags=5):
    """Diebold-Mariano: d = L_a - L_b; t-statistic HAC pentru media lui d (negativ: A mai bun)."""
    d = np.asarray(loss_a) - np.asarray(loss_b)
    ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    t = float(ols.tvalues[0])
    return {'mean_diff': float(d.mean()), 't': t, 'p': 2 * (1 - stats.norm.cdf(abs(t)))}

## Helper functions

In [ ]:
MODELS_OOS = [('GARCH', 'normal', 'GARCH-N'), ('GARCH', 't', 'GARCH-t'), ('GJR', 't', 'GJR-t'), ('EGARCH', 't', 'EGARCH-t')]

INF = {}

from scipy import integrate
from arch.bootstrap import MCS

def oos_forecasts(k='sp500', first_year=2016, H=22):
    r = rets[k]
    idx = r.index
    last_year = idx[-1].year
    out1 = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS}
    outH = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS if lab != 'EGARCH-t'}
    for y in range(first_year, last_year + 1):
        start = idx[idx < f'{y}-01-01'][-1]
        end = idx[idx <= f'{y}-12-31'][-1]
        orig = idx[(idx >= start) & (idx < end)]
        for vol, dist, lab in MODELS_OOS:
            f = fit_garch(r, vol, dist, last_obs=idx[idx >= f'{y}-01-01'][0])
            hh = 1 if vol == 'EGARCH' else H
            fc = f.res.forecast(horizon=hh, start=start, reindex=False).variance / f.c ** 2
            fc = fc.loc[orig]
            pos = idx.get_indexer(fc.index)
            out1[lab].iloc[pos + 1] = fc.iloc[:, 0].values
            if hh == H and lab in outH:
                outH[lab].loc[fc.index] = fc.sum(axis=1).values      # varianta cumulata pe t+1..t+H, la originea t
    ew = ewma_variance(r)
    hist = (r ** 2).rolling(20).mean().shift(1)
    out1['EWMA'] = ew.reindex(idx)
    out1['Hist-20'] = hist
    outH['EWMA'] = (ew.shift(-1) * H).reindex(idx)       # prognoza EWMA este constanta in orizont
    f1 = pd.DataFrame(out1).loc[f'{first_year}-01-01':].dropna()
    fH = pd.DataFrame(outH).dropna()
    return f1, fH

## Analysis

In [ ]:
def qmle_efficiency():
    """Gaussian QMLE: avar = (kappa_z - 1) J^{-1}; clasic (inversa hessianei) = 2 J^{-1}; raport e.s. sqrt((kappa_z-1)/2)."""
    out = {}
    for k in ['sp500', 'btc']:
        f = fit_garch(rets[k], 'GARCH', 'normal')
        cl = f.res.model.fit(disp='off', cov_type='classic', starting_values=f.res.params.values)
        z = f.z.dropna()
        kz = float(stats.kurtosis(z, fisher=False))
        out[k] = {'kappa_z': kz, 'theory_ratio': float(np.sqrt((kz - 1) / 2)),
                  'ratio_alpha': float(f.se['alpha[1]'] / cl.std_err['alpha[1]']),
                  'ratio_beta': float(f.se['beta[1]'] / cl.std_err['beta[1]']),
                  'alpha': float(f.params['alpha[1]']), 'beta': float(f.params['beta[1]'])}
        # stabilitatea aplatizarii de selectie: prima jumatate vs tot esantionul
        out[k]['kappa_half'] = float(stats.kurtosis(z.iloc[:len(z) // 2], fisher=False))
    INF['qmle'] = out


def boundary_tests():
    """LR pentru Normal (1/nu = 0) fata de Student-t: 1/nu >= 0, deci limita este 1/2 chi2(0) + 1/2 chi2(1)."""
    out = {'crit_mix': float(stats.chi2.ppf(0.90, 1)), 'crit_chi2': float(stats.chi2.ppf(0.95, 1))}
    for k in ['sp500', 'btc']:
        fn, ft = fit_garch(rets[k], 'GARCH', 'normal'), fit_garch(rets[k], 'GARCH', 't')
        lr = 2 * (ft.loglik - fn.loglik)
        out[k] = {'lr': float(lr), 'p_mix': float(0.5 * stats.chi2.sf(lr, 1)), 'nu': float(ft.params['nu'])}
    g = fit_garch(rets['sp500'], 'GJR', 't')
    out['gjr_alpha'] = float(g.params['alpha[1]'])
    INF['boundary'] = out


def delta_method():
    """Metoda delta din covarianta robusta a lui (omega, alpha, beta), GARCH(1,1)-t, S&P 500."""
    f = fit_garch(rets['sp500'], 'GARCH', 't')
    p = f.params
    V = f.res.param_cov.loc[['omega', 'alpha[1]', 'beta[1]'], ['omega', 'alpha[1]', 'beta[1]']].values / f.c ** 2
    om, a, b = p['omega'], p['alpha[1]'], p['beta[1]']
    pers = a + b
    se_p = float(np.sqrt(V[1:, 1:].sum()))
    s2 = om / (1 - pers)
    g = np.array([1 / (1 - pers), om / (1 - pers) ** 2, om / (1 - pers) ** 2])
    se_s2 = float(np.sqrt(g @ V @ g))
    vol = np.sqrt(PPY['sp500'] * s2)
    se_vol = PPY['sp500'] * se_s2 / (2 * vol)
    hl = half_life(pers)
    dh = -np.log(0.5) / (pers * np.log(pers) ** 2)
    lo, hi = pers - 1.96 * se_p, pers + 1.96 * se_p
    INF['delta'] = {'pers': float(pers), 'se_p': se_p, 's2': float(s2), 'se_s2': se_s2, 'vol': float(vol),
                    'se_vol': float(se_vol), 'hl': float(hl), 'dh': float(dh), 'se_hl': float(dh * se_p),
                    'p_lo': float(lo), 'p_hi': float(hi), 'hl_lo': float(half_life(lo)),
                    'grad_om': float(g[0]), 'grad_ab': float(g[1])}


def moments_garch_n():
    """GARCH(1,1) cu inovatii Normale: (alpha+beta)^2 + 2 alpha^2, K si rho_1(eps^2), la precizie completa si rotunjit."""
    f = fit_garch(rets['sp500'], 'GARCH', 'normal')
    a, b = f.params['alpha[1]'], f.params['beta[1]']

    def mom(a, b):
        c = (a + b) ** 2 + 2 * a ** 2
        K = 3 * (1 - (a + b) ** 2) / (1 - c)
        rho1 = a * (1 - b ** 2 - a * b) / (1 - b ** 2 - 2 * a * b)
        return float(c), float(K), float(rho1)
    c, K, rho1 = mom(a, b)
    c3, K3, rho13 = mom(round(a, 3), round(b, 3))
    e2 = (rets['sp500'] - rets['sp500'].mean()) ** 2
    INF['mom'] = {'alpha': float(a), 'beta': float(b), 'cond': c, 'K': K, 'rho1': rho1,
                  'cond3': c3, 'K3': K3, 'rho13': rho13, 'rho10': float(rho1 * (a + b) ** 9),
                  'dKda': float((K3 - K)), 'rho1_sample': float(e2.autocorr(1)), 'rho10_sample': float(e2.autocorr(10))}


def li_mak(M=10):
    """Autocorelatiile patratelor reziduurilor standardizate dupa QMLE Gaussian (S&P 500, GARCH-N).
    sqrt(n) r_hat -> N(0, C), C = I - H' J^{-1} H / (kappa_z - 1), cu J = E[g g'], H_k = E[g_t (z_{t-k}^2 - 1)],
    g_t = d ln sigma_t^2 / d(omega, alpha, beta) (derivarea lui Li & Mak, 1994, pentru parametrii dispersiei)."""
    f = fit_garch(rets['sp500'], 'GARCH', 'normal')
    p = f.params
    e = (rets['sp500'] - p['mu']).values
    s2 = (f.sigma ** 2).values
    n = len(e)
    e2l = np.concatenate(([np.mean(e ** 2)], e[:-1] ** 2))
    s2l = np.concatenate(([np.mean(e ** 2)], s2[:-1]))
    D = np.column_stack([lfilter([1.0], [1.0, -p['beta[1]']], x) for x in (np.ones(n), e2l, s2l)])
    g = D / s2[:, None]
    z2 = e ** 2 / s2
    u = z2 - 1
    s2u = np.mean(u ** 2)
    J = g.T @ g / n
    H = np.array([(g[k:] * u[:-k, None]).mean(axis=0) for k in range(1, M + 1)])     # M x 3
    C = np.eye(M) - H @ np.linalg.solve(J, H.T) / s2u
    rh = np.array([np.sum(u[k:] * u[:-k]) / np.sum(u ** 2) for k in range(1, M + 1)])
    q_lb = n * (n + 2) * np.sum(rh ** 2 / (n - np.arange(1, M + 1)))
    q_lm = n * rh @ np.linalg.solve(C, rh)
    INF['limak'] = {'M': M, 'c11': float(C[0, 0]), 'c55': float(C[4, 4]), 'cMM': float(C[M - 1, M - 1]),
                    'q_lb': float(q_lb), 'p_lb': float(stats.chi2.sf(q_lb, M)), 'q_lm': float(q_lm),
                    'p_lm': float(stats.chi2.sf(q_lm, M)), 'r1': float(rh[0]), 'trace_loss': float(M - np.trace(C))}


def gjr_skew_persistence():
    """GJR-GARCH cu inovatii t asimetrice (Hansen, 1994), S&P 500: E[z^2 1(z<0)] in locul lui 1/2."""
    f = fit_garch(rets['sp500'], 'GJR', 'skewt')
    d = f.res.model.distribution
    par = f.res.params[d.parameter_names()].values
    pdf = lambda x: float(np.exp(d.loglikelihood(par, np.array([x]), np.array([1.0]), individual=True))[0])
    m_neg = integrate.quad(lambda x: x ** 2 * pdf(x), -np.inf, 0, limit=200)[0]
    p_neg = integrate.quad(pdf, -np.inf, 0, limit=200)[0]
    p = f.params
    INF['gjrskew'] = {'alpha': float(p['alpha[1]']), 'gamma': float(p['gamma[1]']), 'beta': float(p['beta[1]']),
                      'eta': float(par[0]), 'lam': float(par[1]), 'm_neg': float(m_neg), 'p_neg': float(p_neg),
                      'pers_sym': float(p['alpha[1]'] + p['gamma[1]'] / 2 + p['beta[1]']),
                      'pers_skew': float(p['alpha[1]'] + p['gamma[1]'] * m_neg + p['beta[1]'])}


def oos_rolling(k='sp500', first_year=2016, W=1000):
    """Prognoze pe o zi cu fereastra MOBILA de W zile, reestimata in fiecare ianuarie (cadrul Giacomini-White)."""
    r = rets[k]
    idx = r.index
    out = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS}
    for y in range(first_year, idx[-1].year + 1):
        start = idx[idx < f'{y}-01-01'][-1]
        end = idx[idx <= f'{y}-12-31'][-1]
        orig = idx[(idx >= start) & (idx < end)]
        pos0 = idx.get_loc(start)
        sub = r.iloc[pos0 - W + 1:]
        for vol, dist, lab in MODELS_OOS:
            f = fit_garch(sub, vol, dist, last_obs=idx[pos0 + 1])
            fc = f.res.forecast(horizon=1, start=start, reindex=False).variance / f.c ** 2
            fc = fc.loc[orig]
            pos = idx.get_indexer(fc.index)
            out[lab].iloc[pos + 1] = fc.iloc[:, 0].values
    out['EWMA'] = ewma_variance(r).reindex(idx)
    out['Hist-20'] = (r ** 2).rolling(20).mean().shift(1)
    return pd.DataFrame(out).loc[f'{first_year}-01-01':].dropna()


def gw_test(la, lb):
    """Giacomini-White (2006), test conditional, orizont 1: instrumente h_t = (1, d_t); GW = n Zbar' Omega^{-1} Zbar ~ chi2(2)."""
    d = np.asarray(la) - np.asarray(lb)
    Z = np.column_stack([d[1:], d[:-1] * d[1:]])
    zb = Z.mean(axis=0)
    Om = Z.T @ Z / len(Z)
    stat = len(Z) * zb @ np.linalg.solve(Om, zb)
    return float(stat), float(stats.chi2.sf(stat, 2))


def gw_mcs(k='sp500'):
    r = rets[k]
    fr = oos_rolling(k)
    proxy = (r ** 2).reindex(fr.index)
    L = pd.DataFrame({m: qlike(proxy, fr[m]) for m in fr.columns})
    rows = {}
    for m in L.columns:
        if m == 'GARCH-t':
            continue
        dm = dm_test(L[m], L['GARCH-t'])
        gw, gp = gw_test(L[m], L['GARCH-t'])
        rows[m] = {'uncond_t': dm['t'], 'gw': gw, 'gw_p': gp, 'mean_diff': dm['mean_diff']}
    # MCS pe prognozele cu fereastra in expansiune (tabelul din curs / seminar)
    f1, _ = oos_forecasts(k, 2016, 1)
    px = (r ** 2).reindex(f1.index)
    Le = pd.DataFrame({m: qlike(px, f1[m]) for m in f1.columns})
    mcs = MCS(Le, size=0.10, reps=10000, block_size=10, method='R', bootstrap='stationary', seed=SEED)
    mcs.compute()
    pv = mcs.pvalues['Pvalue']
    INF[f'gw_{k}'] = {'n': len(fr), 'W': 1000, 'qlike_roll': L.mean().to_dict(), 'tests': rows,
                      'mcs_p': pv.to_dict(), 'mcs_in': list(mcs.included), 'n_mcs': len(Le)}

## Run

In [ ]:
qmle_efficiency(); boundary_tests(); delta_method(); moments_garch_n(); li_mak(); gjr_skew_persistence()
gw_mcs('sp500'); gw_mcs('bet')
print(json.dumps(INF, indent=1, default=float))